In [ ]:
import pandas as pd
import plotly.graph_objects as go

In [ ]:
theses = pd.read_csv(r"../data/theses.csv")
# Add new column for the number of supervisors which is the number of not empty "directeurs_these.i.nom"
theses["nb_sup"] = theses[
    ["directeurs_these.0.nom", "directeurs_these.1.nom", "directeurs_these.2.nom", "directeurs_these.3.nom",
     "directeurs_these.4.nom", "directeurs_these.5.nom", "directeurs_these.6.nom"]].notna().sum(axis=1)

## Make dataframe

### Original ddc to discipline mapping

In [ ]:
og_dict = {"COMP": ["000", "004"],
           "PSYC": ["020", "060", "070", "090", "100", "110", "120", "130", "140", "150", "160", "170", "180", "190",
                    "200",
                    "210", "220", "230", "240", "250", "260", "270", "280", "290"],
           "SOCI": ["300", "350", "360", "370", "380", "390"],
           "ECON": ["310", "320", "330", "340"],
           "ARTS": ["400", "410", "420", "430", "440", "450", "460", "470", "480", "490", "700", "710", "720", "730",
                    "740",
                    "750", "760", "770", "780", "790", "800", "810", "820", "830", "840", "850", "860", "870", "880",
                    "890",
                    "900", "910", "920", "930", "940", "944", "950", "960", "970", "980", "990"],
           "MATH": ["500", "510"],
           "PHYS": ["520", "530"],
           "CHEM": ["540"],
           "EART": ["550", "560"],
           "BIOC": ["570", "580", "590"],
           "ENGI": ["600", "620", "670", "680", "690"],
           "MEDI": ["610", "796"],
           "VETE": ["630"],
           "DECI": ["640"],
           "BUSI": ["650"],
           "CENGI": ["660"]
           }

d = {}
for word, values in og_dict.items():
    for value in values:
        d[value] = word

### Clean ids

In [ ]:
nb_directeurs = 6
director_ids = [f"directeurs_these.{i}.idref" for i in range(0, nb_directeurs + 1)]

# Students with no idref are assigned value of -1 * index
theses.loc[theses["auteur.idref"].isna(), "auteur.idref"] = (
        "P" + pd.Series(theses.index[theses["auteur.idref"].isna()]).astype(str)).values
# Directors with no idref are assigned of -1 * (own index + 1 + index of the student)
for i in range(0, nb_directeurs + 1):
    mask = theses[f"directeurs_these.{i}.idref"].isna() & (
            theses[f"directeurs_these.{i}.nom"].notna() | theses[f"directeurs_these.{i}.prenom"].notna()
    )
    theses.loc[mask, f"directeurs_these.{i}.idref"] = "D" + pd.Series(i + 1 + theses.index[mask]).astype(str)

### Clean the dataframe

In [ ]:
students = theses[["auteur.idref", "auteur.nom", "auteur.prenom", "date_soutenance", "oai_set_specs", "nb_sup"]]
students.columns = ["idref", "nom", "prenom", "date_soutenance", "discipline", "nb_sup"]
# Clean the discipline column by removing the prefix "ddc:" and assigning the corresponding discipline based on the mapping
students["discipline"] = students["discipline"].str.replace("ddc:", "")
# If discipline contains "||", assign "MULT", otherwise map to the corresponding discipline or None
students["discipline"] = students["discipline"].apply(
    lambda x: "MULT" if type(x) is str and "||" in x else d.get(x, None))
students["date_soutenance"] = students["date_soutenance"].astype(str).str.split('-').str[0]
# Remove all "nan" value in date_soutenance
students = students[students["date_soutenance"] != "nan"]
students.to_parquet(r"../data/students.parquet", engine="pyarrow")

## Make figures

In [ ]:
# Drop columns with no date
time_evolution = students.dropna(subset=["date_soutenance"])
# Write only years in date_soutenance
time_evolution["date_soutenance"] = time_evolution["date_soutenance"].astype(str).str.split('-').str[0]
# Group by year and count the number of theses
overall_time_evolution = time_evolution.groupby("date_soutenance").size().reset_index(name="count")
# Group by year and discipline and count the number of theses
per_disc_time_evolution = time_evolution.groupby(["date_soutenance", "discipline"]).size().reset_index(name="count")
# Group by year according to number of supervisors
per_sup_time_evolution = time_evolution.groupby(["date_soutenance", "nb_sup"]).size().reset_index(name="count")
overall_time_evolution.to_parquet(r"../data/overall_time_evolution.parquet", engine="pyarrow")
per_disc_time_evolution.to_parquet(r"../data/per_disc_time_evolution.parquet", engine="pyarrow")
per_sup_time_evolution.to_parquet(r"../data/per_sup_time_evolution.parquet", engine="pyarrow")
figure = go.Figure()
figure.add_trace(go.Scatter(
    x=overall_time_evolution["date_soutenance"],
    y=overall_time_evolution["count"],
    connectgaps=True,
    mode='lines+markers',
    name='Overall',
    line=dict(color='blue')
))
figure.add_trace(go.Scatter(
    x=per_disc_time_evolution["date_soutenance"],
    y=per_disc_time_evolution["count"],
    connectgaps=True,
    mode='lines+markers',
    name='Per Discipline',
    line=dict(color='red')
))
figure.add_trace(go.Scatter(
    x=per_sup_time_evolution["date_soutenance"],
    y=per_sup_time_evolution["count"],
    connectgaps=True,
    mode='lines+markers',
    name='Per Supervisor',
    line=dict(color='green')
))
figure.update_layout(
    title="Temporal Evolution of Theses",
    xaxis_title="Year",
    yaxis_title="Number of Theses",
    showlegend=True,
    template="plotly"
)
figure.write_image(f"./results_new/temporal_evolution.png")
figure.write_html(f"./results_new/temporal_evolution.html")
figure.show()

## Make figures for discipline evolution

### Exploded dataframe (disciplines are no longer cumulative)

In [ ]:
import pandas as pd

theses = pd.read_csv(r"C:\Users\sayfe\Desktop\PER\MultidisciplinaryPhD_Aurora\data\raw\theses\theses-soutenues.csv")
# Retrieve all columns with name containing "idref"
columns = theses.columns.tolist().copy()
filtered_columns = [column for column in columns if "idref" in column]
# Change nan to empty
for column in filtered_columns:
    theses[column] = theses[column].astype(str)
# shorten date_soutenance to only the year
theses["date_soutenance"] = theses["date_soutenance"].astype(str).str.split('-').str[0]
theses_copy = theses.copy(deep=True)
# drop line with date equal to nan
theses = theses[theses["date_soutenance"] != "nan"]
# transform value in oai into arrays
theses["oai_set_specs"] = theses["oai_set_specs"].apply(lambda oai: str(oai).split("||"))
theses = theses.explode("oai_set_specs")

theses.to_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-explode.parquet",
                  engine="pyarrow")

## Evolution of disciplines

In [ ]:
theses = pd.read_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-explode.parquet",
                         engine="pyarrow")

### Mapping ddc to extended disciplines

In [ ]:
ddc_dict = {
    "ddc:000": "Informatique, information, généralités",
    "ddc:004": "Informatique",
    "ddc:020": "Bibliothéconomie et sciences de l'information",
    "ddc:060": "Organisations générales et muséologie",
    "ddc:070": "Médias d'information, journalisme, édition",
    "ddc:090": "Manuscrits et livres rares",
    "ddc:100": "Philosophie, psychologie",
    "ddc:110": "Métaphysique",
    "ddc:120": "Epistémologie, causalité, genre humain",
    "ddc:130": "Phénomènes paranormaux, seudosciences",
    "ddc:140": "Les divers systèmes et écoles philosophiques",
    "ddc:150": "Psychologie",
    "ddc:160": "Logique",
    "ddc:170": "Morale (éthique)",
    "ddc:180": "Philosophie de l'Antiquité, du Moyen Âge, de l'Orient",
    "ddc:190": "Philosophie occidentale moderne et philosophies non orientales",
    "ddc:200": "Religion",
    "ddc:210": "Philosophie et théorie de la religion",
    "ddc:220": "Bible",
    "ddc:230": "Théologie chrétienne",
    "ddc:240": "Théologie morale et pratiques chrétiennes",
    "ddc:250": "Eglises locales, ordres religieux chrétiens",
    "ddc:260": "Théologie chrétienne et société, ecclésiologie",
    "ddc:270": "Histoire et géographie du christianisme et de l'Eglise chrétienne",
    "ddc:280": "Confessions et sectes de l'Eglise chrétienne",
    "ddc:290": "Autres religions",
    "ddc:300": "Sciences sociales, sociologie, anthropologie",
    "ddc:310": "Statistiques générales",
    "ddc:320": "Science politique",
    "ddc:330": "Economie",
    "ddc:340": "Droit",
    "ddc:350": "Administration publique. Arts et science militaires",
    "ddc:360": "Problèmes et services sociaux",
    "ddc:370": "Education et enseignement",
    "ddc:380": "Commerce, communications, transports",
    "ddc:390": "Ethnologie",
    "ddc:400": "Langues et linguistique",
    "ddc:410": "Linguistique générale",
    "ddc:420": "Langue anglaise. Anglo-saxon",
    "ddc:430": "Langues germaniques. Allemand",
    "ddc:440": "Langues romanes. Français",
    "ddc:450": "Langues italienne, roumaine, rhéto-romane",
    "ddc:460": "Langues espagnole et portugaise",
    "ddc:470": "Langues italiques. Latin",
    "ddc:480": "Langues helléniques. Grec classique",
    "ddc:490": "Autres langues",
    "ddc:500": "Sciences de la nature et mathématiques",
    "ddc:510": "Mathématiques",
    "ddc:520": "Astronomie, cartographie, géodésie",
    "ddc:530": "Physique",
    "ddc:540": "Chimie, minéralogie, cristallographie",
    "ddc:550": "Sciences de la terre",
    "ddc:560": "Paléontologie. Paléozoologie",
    "ddc:570": "Sciences de la vie, biologie, biochimie",
    "ddc:580": "Plantes. Botanique",
    "ddc:590": "Animaux. Zoologie",
    "ddc:600": "Technologie (Sciences appliquées)",
    "ddc:610": "Médecine et santé",
    "ddc:620": "Sciences de l'ingénieur",
    "ddc:630": "Agronomie, agriculture et médecine vétérinaire",
    "ddc:640": "Economie domestique. Vie familiale",
    "ddc:650": "Gestion et organisation de l'entreprise",
    "ddc:660": "Génie chimique, technologies alimentaires",
    "ddc:670": "Fabrication industrielle",
    "ddc:680": "Fabrication de produits à usages spécifiques",
    "ddc:690": "Bâtiments",
    "ddc:700": "Arts. Beaux-arts et arts décoratifs",
    "ddc:710": "Urbanisme",
    "ddc:720": "Architecture",
    "ddc:730": "Arts plastiques. Sculpture",
    "ddc:740": "Dessin. Arts décoratifs",
    "ddc:750": "Peinture",
    "ddc:760": "Arts graphiques",
    "ddc:770": "Photographie et les photographies, art numérique",
    "ddc:780": "Musique",
    "ddc:790": "Arts du spectacle, loisirs",
    "ddc:796": "Sport",
    "ddc:800": "Histoire et critique littéraires, rhétorique",
    "ddc:810": "Littérature américaine en anglais",
    "ddc:820": "Littératures anglaise et anglo-saxonne",
    "ddc:830": "Littérature allemande",
    "ddc:840": "Littérature de langues romanes. Littérature française",
    "ddc:850": "Littérature italienne",
    "ddc:860": "Littératures espagnole et portugaise",
    "ddc:870": "Littérature latine",
    "ddc:880": "Littérature grecque",
    "ddc:890": "Littératures des autres langues",
    "ddc:900": "Géographie et histoire",
    "ddc:910": "Géographie et voyages",
    "ddc:920": "Biographies générales, généalogie, emblèmes",
    "ddc:930": "Histoire ancienne et préhistoire",
    "ddc:940": "Histoire moderne et contemporaine de l'Europe",
    "ddc:944": "Histoire générale de la France",
    "ddc:950": "Histoire générale de l'Asie, Orient, Extrême-Orient",
    "ddc:960": "Histoire générale de l'Afrique",
    "ddc:970": "Histoire générale de l'Amérique du Nord",
    "ddc:980": "Histoire générale de l'Amérique du Sud",
    "ddc:990": "Histoire générale des autres parties du monde, des mondes extraterrestres. Iles du Pacifique"
}

In [ ]:
# revert dict
# disc_to_ddc_dict = {v: k for k, v in ddc_dict.items()}
# map ddc to discipline
def map_ddc_to_discipline(ddc):
    return ddc_dict.get(ddc, None)

# map discipline to ddc
# def map_discipline_to_ddc(discipline):
#     return disc_to_ddc_dict.get(discipline, None)

### New ddc to SCOPUS discipline mapping

In [ ]:
ddc_scopus_new = {
    "ddc:000": "COMP",
    "ddc:004": "COMP",
    "ddc:020": "SOCI",
    "ddc:060": "ARTS",
    "ddc:070": "SOCI",
    "ddc:090": "ARTS",
    "ddc:100": "PSYC",
    "ddc:110": "ARTS",
    "ddc:120": "ARTS",
    "ddc:130": "PSYC",
    "ddc:140": "ARTS",
    "ddc:150": "PSYC",
    "ddc:160": "MATH",
    "ddc:170": "ARTS",
    "ddc:180": "ARTS",
    "ddc:190": "ARTS",
    "ddc:200": "ARTS",
    "ddc:210": "ARTS",
    "ddc:220": "ARTS",
    "ddc:230": "ARTS",
    "ddc:240": "ARTS",
    "ddc:250": "ARTS",
    "ddc:260": "ARTS",
    "ddc:270": "ARTS",
    "ddc:280": "ARTS",
    "ddc:290": "ARTS",
    "ddc:300": "SOCI",
    "ddc:310": "DECI",
    "ddc:320": "SOCI",
    "ddc:330": "ECON",
    "ddc:340": "SOCI",
    "ddc:350": "SOCI",
    "ddc:360": "SOCI",
    "ddc:370": "SOCI",
    "ddc:380": "SOCI",
    "ddc:390": "SOCI",
    "ddc:400": "ARTS",
    "ddc:410": "ARTS",
    "ddc:420": "ARTS",
    "ddc:430": "ARTS",
    "ddc:440": "ARTS",
    "ddc:450": "ARTS",
    "ddc:460": "ARTS",
    "ddc:470": "ARTS",
    "ddc:480": "ARTS",
    "ddc:490": "ARTS",
    "ddc:500": "MATH",
    "ddc:510": "MATH",
    "ddc:520": "PHYS",
    "ddc:530": "PHYS",
    "ddc:540": "CHEM",
    "ddc:550": "EART",
    "ddc:560": "EART",
    "ddc:570": "BIOC",
    "ddc:580": "AGRI",
    "ddc:590": "AGRI",
    "ddc:600": "ENGI",
    "ddc:610": "MEDI",
    "ddc:620": "ENGI",
    "ddc:630": "VETE",
    "ddc:640": "ECON",
    "ddc:650": "BUSI",
    "ddc:660": "CENGI",
    "ddc:670": "ENGI",
    "ddc:680": "ENGI",
    "ddc:690": "ENGI",
    "ddc:700": "ARTS",
    "ddc:710": "ENGI",
    "ddc:720": "ENGI",
    "ddc:730": "ARTS",
    "ddc:740": "ARTS",
    "ddc:750": "ARTS",
    "ddc:760": "ARTS",
    "ddc:770": "ARTS",
    "ddc:780": "ARTS",
    "ddc:790": "ARTS",
    "ddc:796": "HEAL",
    "ddc:800": "ARTS",
    "ddc:810": "ARTS",
    "ddc:820": "ARTS",
    "ddc:830": "ARTS",
    "ddc:840": "ARTS",
    "ddc:850": "ARTS",
    "ddc:860": "ARTS",
    "ddc:870": "ARTS",
    "ddc:880": "ARTS",
    "ddc:890": "ARTS",
    "ddc:900": "SOCI",
    "ddc:910": "SOCI",
    "ddc:920": "ARTS",
    "ddc:930": "ARTS",
    "ddc:940": "ARTS",
    "ddc:944": "ARTS",
    "ddc:950": "ARTS",
    "ddc:960": "ARTS",
    "ddc:970": "ARTS",
    "ddc:980": "ARTS",
    "ddc:990": "ARTS"
}

# map ddc to new scopus
def map_ddc_to_discipline_scopus_new(ddc):
    to_return = ddc_scopus_new.get(ddc, None)
    if to_return is None:
        print("Not found: ", ddc)
    return to_return

### Apply the mapping function to the discipline column, build and plot the graph

In [ ]:
# Apply the mapping function to the discipline column
theses["oai_set_specs"] = theses["oai_set_specs"].apply(map_ddc_to_discipline_scopus_new)
# Remove rows with no discipline
# theses = theses[theses["discipline"].notna()]

# Keep only idref nom prenom date_soutencance discipline
theses = theses[["auteur.idref", "auteur.nom", "auteur.prenom", "date_soutenance", "oai_set_specs"]]
# Save the dataframe
theses.to_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-explode-disc-new-match.parquet",
                  engine="pyarrow")

In [ ]:
# retrieve colors from plotly
import plotly.express as px

colors = (px.colors.qualitative.Alphabet + px.colors.qualitative.Light24 + px.colors.qualitative.Set2
          + px.colors.qualitative.Set1 + px.colors.qualitative.Set3 + px.colors.qualitative.Plotly
          + px.colors.qualitative.Prism)
len(colors)

In [ ]:
# order dict by values
ddc_dict = dict(sorted(ddc_dict.items(), key=lambda item: item[1]))

In [ ]:
# build line graph for number of occurrences of each discipline per ordered years
theses.sort_values(by=["date_soutenance"], inplace=True)
figure = go.Figure()
for i, (discipline, color) in enumerate(zip(ddc_scopus_new.values(), colors[:len(ddc_scopus_new)])):
    # Filter the dataframe for the current discipline
    discipline_df = theses[theses["oai_set_specs"] == discipline]
    # Group by year and count the number of theses
    discipline_time_evolution = discipline_df.groupby("date_soutenance").size().reset_index(name="count")
    figure.add_trace(go.Scatter(
        x=discipline_time_evolution["date_soutenance"],
        y=discipline_time_evolution["count"],
        connectgaps=True,
        mode='lines+markers',
        name=discipline,
        line=dict(color=color)
    ))

toggle_button = dict(
    type="buttons",
    direction="right",
    x=1,
    y=1,
    xanchor="right",
    yanchor="top",
    pad=dict(r=10, t=10),
    buttons=[
        dict(
            label="Show Legend",
            method="relayout",
            args=[{"showlegend": True}]
        ),
        dict(
            label="Hide Legend",
            method="relayout",
            args=[{"showlegend": False}]
        )
    ]
)

figure.update_layout(
    height=700,
    showlegend=False,
    updatemenus=[toggle_button],
    title="Temporal Evolution of Disciplines SCOPUS NEW",
    xaxis_title="Year",
    yaxis_title="Number of Theses",
    template="plotly"
)
figure.update_xaxes(categoryorder='category ascending')
# order legend alphabetically

figure.write_image(f"./results_new/temporal_evolution_disciplines_scopus_new.png")
figure.write_html(f"./results_new/temporal_evolution_disciplines_scopus_new.html")
figure.show()

## Make figures for students evolution

In [ ]:
students = pd.read_parquet(r"../data/students.parquet", engine="pyarrow")

disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENG', 'CHEM' 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI',
               'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']

# build line graph for number of occurrences of each discipline per ordered years
figure = go.Figure()
for i, (discipline, color) in enumerate(zip(disciplines, colors)):
    # Filter the dataframe for the current discipline
    discipline_df = students[students["discipline"] == discipline]
    # Group by year and count the number of theses
    discipline_time_evolution = discipline_df.groupby("date_soutenance").size().reset_index(name="count")
    figure.add_trace(go.Scatter(
        x=discipline_time_evolution["date_soutenance"],
        y=discipline_time_evolution["count"],
        connectgaps=True,
        mode='lines+markers',
        name=discipline,
        line=dict(color=color)
    ))
toggle_button = dict(
    type="buttons",
    direction="right",
    x=1,
    y=1,
    xanchor="right",
    yanchor="top",
    pad=dict(r=10, t=10),
    buttons=[
        dict(
            label="Show Legend",
            method="relayout",
            args=[{"showlegend": True}]
        ),
        dict(
            label="Hide Legend",
            method="relayout",
            args=[{"showlegend": False}]
        )
    ]
)

figure.update_layout(
    height=700,
    showlegend=False,
    updatemenus=[toggle_button],
    title="Temporal Evolution of Disciplines SCOPUS",
    xaxis_title="Year",
    yaxis_title="Number of Theses",
    template="plotly"
)
figure.update_xaxes(categoryorder='category ascending')
figure.write_image(f"./results_new/temporal_evolution_students_disciplines.png")
figure.write_html(f"./results_new/temporal_evolution_students_disciplines.html")
figure.show()

In [ ]:
df = pd.read_parquet(r"../data/theses-explode-disc-new-match.parquet", engine="pyarrow")

In [ ]:
# merge duplicates based on nnt, make oai_set_specs a set
df = df.groupby(["auteur.idref", "auteur.nom", "auteur.prenom", "date_soutenance"]).agg(
    {"oai_set_specs": lambda x: set(x)}).reset_index()

In [ ]:
# add new column as boolean for whether oai_set_specs len > 1
df["multi"] = df["oai_set_specs"].apply(lambda x: len(x) > 1)

In [ ]:
# save the dataframe
df.to_parquet(r"../data/students_multi.parquet", engine="pyarrow")

In [ ]:
# count the number of theses with multi disciplines
multi_count = df["multi"].sum()

In [ ]:
multi_count

In [ ]:
new_df = df[df["multi"] == True]

In [ ]:
import pandas as pd
new_df =pd.read_parquet(r"../data/multi_disc_students.parquet", engine="pyarrow")

In [ ]:
# count the number of theses with exactly 2 disciplines
two_count = new_df[new_df["oai_set_specs"].apply(lambda x: len(x) == 2)].shape[0]
two_count

In [ ]:
# drop all rows with more than 2 disciplines
new_df = new_df[new_df["oai_set_specs"].apply(lambda x: len(x) <= 2)]

In [ ]:
len(new_df)

In [ ]:
# save the dataframe
new_df.to_parquet(r"../data/multi_disc_students_2.parquet", engine="pyarrow", index=False)

In [ ]:
new_df.head(2)

In [ ]:
import plotly.express as px
disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENGI', 'CHEM', 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI',
               'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']
colors = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Set3
# build dictionary
disciplines_dict = {}
for i, discipline in enumerate(disciplines):
    disciplines_dict[discipline] = colors[i]

print(disciplines_dict)

In [ ]:
# transform oai_set_specs into a lists
new_df["oai_set_specs"] = new_df["oai_set_specs"].apply(lambda x: list(x))
# get all disciplines in the dataframe
all_disciplines = set()
for i in new_df["oai_set_specs"]:
    all_disciplines.add(i[0])
    all_disciplines.add(i[1])

# sort the disciplines
all_disciplines = sorted(list(all_disciplines))
print(all_disciplines)

In [ ]:
new_df.head(2)

In [ ]:
for discipline in all_disciplines:
    print(discipline)
    # Filter the dataframe for the current discipline
    discipline_df = new_df[new_df["oai_set_specs"].apply(lambda x: discipline in x)].copy(deep=True)
    discipline_df["oai_set_specs"] = discipline_df["oai_set_specs"].apply(
        lambda x: [i for i in x if i != discipline][0])
    figure_line = go.Figure()
    figure_bar = go.Figure()
    # Group by year and and oai_set_specs and count the number of theses
    discipline_time_evolution = discipline_df
    discipline_time_evolution["counts"] = 1
    discipline_time_evolution = discipline_df.groupby(["date_soutenance", "oai_set_specs"]).count().reset_index()
    # sort the dataframe by oai_set_specs
    discipline_time_evolution.sort_values(by=["date_soutenance","oai_set_specs"], inplace=True)
    # add one trace per oai_set_specs
    oai_set_specs = discipline_time_evolution["oai_set_specs"].unique()
    # sort the oai_set_specs
    oai_set_specs = sorted(list(oai_set_specs))
    for i, oai_set_spec in enumerate(oai_set_specs):
        trace_df = discipline_time_evolution[discipline_time_evolution["oai_set_specs"] == oai_set_spec]
        figure_line.add_trace(go.Scatter(
            x=trace_df["date_soutenance"],
            y=trace_df["counts"],
            connectgaps=False,
            mode='lines+markers',
            name=f"{discipline} - {oai_set_spec}",
            line=dict(color=disciplines_dict[oai_set_spec])
        ))
        figure_bar.add_trace(go.Bar(
            x=trace_df["date_soutenance"],
            y=trace_df["counts"],
            name=f"{discipline} - {oai_set_spec}",
            marker_color=disciplines_dict[oai_set_spec]
        ))
    figure_line.update_layout(
        height=700,
        title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
        xaxis_title="Year",
        yaxis_title="Number of Theses",
        template="plotly"
    )
    figure_line.update_xaxes(categoryorder='category ascending')
    figure_line.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}.png")
    figure_line.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}.html")
    figure_bar.update_layout(
        height=700,
        title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
        xaxis_title="Year",
        yaxis_title="Number of Theses",
        template="plotly"
    )
    figure_bar.update_xaxes(categoryorder='category ascending')
    figure_bar.update_layout(barmode='stack')
    figure_bar.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}_bar.png")
    figure_bar.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}_bar.html")


## Build new files and graphs

In [ ]:
import pandas as pd
theses = pd.read_csv(r"C:\Users\sayfe\Desktop\PER\MultidisciplinaryPhD_Aurora\data\raw\theses\theses-soutenues.csv")

In [ ]:
# theses.columns

### Make lighter dataframe

In [ ]:
# columns with directeur
directeur = [column for column in theses.columns if "directeurs_these" in column]
# columns with auteur
auteur = [column for column in theses.columns if "auteur" in column]
theses = theses[auteur + directeur + ["date_soutenance", "oai_set_specs", "nnt"]]

In [ ]:
# make idref columns string type
for column in theses.columns:
    if "idref" in column:
        theses[column] = theses[column].astype(str)

In [ ]:
theses.to_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-light.parquet", engine="pyarrow")

In [ ]:
len(theses), len(theses["nnt"])

In [ ]:
# assign value of "P" + index to auteur.idref
theses.loc[theses["auteur.idref"] == "nan", "auteur.idref"] = (
        "P" + pd.Series(theses.index[theses["auteur.idref"] == "nan"]).astype(str)).values

In [ ]:
# count number of supervisors by checking if the idref, nom and prenom are all not nan
def count_supervisors(row):
    count = 0
    for i in range(7):
        if (pd.notna(row[f"directeurs_these.{i}.idref"]) and row[f"directeurs_these.{i}.idref"]!="nan") or \
                (pd.notna(row[f"directeurs_these.{i}.nom"]) and row[f"directeurs_these.{i}.nom"]!="nan") or \
                (pd.notna(row[f"directeurs_these.{i}.prenom"]) and row[f"directeurs_these.{i}.prenom"]!="nan"):
            count += 1
    return count

# apply the function to the dataframe
theses["nb_sup"] = theses.apply(count_supervisors, axis=1)

In [ ]:
# lighter dataframe
theses = theses[auteur + ["date_soutenance", "oai_set_specs", "nb_sup", "nnt"]]

In [ ]:
# save the dataframe
theses.to_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-lighter.parquet", engine="pyarrow")

### OLD ddc to discipline mapping

In [ ]:
og_dict = {"COMP": ["000", "004"],
           "PSYC": ["020", "060", "070", "090", "100", "110", "120", "130", "140", "150", "160", "170", "180", "190",
                    "200",
                    "210", "220", "230", "240", "250", "260", "270", "280", "290"],
           "SOCI": ["300", "350", "360", "370", "380", "390"],
           "ECON": ["310", "320", "330", "340"],
           "ARTS": ["400", "410", "420", "430", "440", "450", "460", "470", "480", "490", "700", "710", "720", "730",
                    "740",
                    "750", "760", "770", "780", "790", "800", "810", "820", "830", "840", "850", "860", "870", "880",
                    "890",
                    "900", "910", "920", "930", "940", "944", "950", "960", "970", "980", "990"],
           "MATH": ["500", "510"],
           "PHYS": ["520", "530"],
           "CHEM": ["540"],
           "EART": ["550", "560"],
           "BIOC": ["570", "580", "590"],
           "ENGI": ["600", "620", "670", "680", "690"],
           "MEDI": ["610", "796"],
           "VETE": ["630"],
           "DECI": ["640"],
           "BUSI": ["650"],
           "CENGI": ["660"]
           }

d = {}
for word, values in og_dict.items():
    for value in values:
        d["ddc:"+value] = word

# mapping function
def map_ddc_to_discipline_scopus(ddc):
    to_return = d.get(ddc, None)
    if to_return is None:
        print("Not found: ", ddc)
    return to_return

### NEW ddc to discipline mapping


In [ ]:
ddc_scopus_new = {
    "ddc:000": "COMP",
    "ddc:004": "COMP",
    "ddc:020": "SOCI",
    "ddc:060": "ARTS",
    "ddc:070": "SOCI",
    "ddc:090": "ARTS",
    "ddc:100": "PSYC",
    "ddc:110": "ARTS",
    "ddc:120": "ARTS",
    "ddc:130": "PSYC",
    "ddc:140": "ARTS",
    "ddc:150": "PSYC",
    "ddc:160": "MATH",
    "ddc:170": "ARTS",
    "ddc:180": "ARTS",
    "ddc:190": "ARTS",
    "ddc:200": "ARTS",
    "ddc:210": "ARTS",
    "ddc:220": "ARTS",
    "ddc:230": "ARTS",
    "ddc:240": "ARTS",
    "ddc:250": "ARTS",
    "ddc:260": "ARTS",
    "ddc:270": "ARTS",
    "ddc:280": "ARTS",
    "ddc:290": "ARTS",
    "ddc:300": "SOCI",
    "ddc:310": "DECI",
    "ddc:320": "SOCI",
    "ddc:330": "ECON",
    "ddc:340": "SOCI",
    "ddc:350": "SOCI",
    "ddc:360": "SOCI",
    "ddc:370": "SOCI",
    "ddc:380": "SOCI",
    "ddc:390": "SOCI",
    "ddc:400": "ARTS",
    "ddc:410": "ARTS",
    "ddc:420": "ARTS",
    "ddc:430": "ARTS",
    "ddc:440": "ARTS",
    "ddc:450": "ARTS",
    "ddc:460": "ARTS",
    "ddc:470": "ARTS",
    "ddc:480": "ARTS",
    "ddc:490": "ARTS",
    "ddc:500": "MATH",
    "ddc:510": "MATH",
    "ddc:520": "PHYS",
    "ddc:530": "PHYS",
    "ddc:540": "CHEM",
    "ddc:550": "EART",
    "ddc:560": "EART",
    "ddc:570": "BIOC",
    "ddc:580": "AGRI",
    "ddc:590": "AGRI",
    "ddc:600": "ENGI",
    "ddc:610": "MEDI",
    "ddc:620": "ENGI",
    "ddc:630": "VETE",
    "ddc:640": "ECON",
    "ddc:650": "BUSI",
    "ddc:660": "CENGI",
    "ddc:670": "ENGI",
    "ddc:680": "ENGI",
    "ddc:690": "ENGI",
    "ddc:700": "ARTS",
    "ddc:710": "ENGI",
    "ddc:720": "ENGI",
    "ddc:730": "ARTS",
    "ddc:740": "ARTS",
    "ddc:750": "ARTS",
    "ddc:760": "ARTS",
    "ddc:770": "ARTS",
    "ddc:780": "ARTS",
    "ddc:790": "ARTS",
    "ddc:796": "HEAL",
    "ddc:800": "ARTS",
    "ddc:810": "ARTS",
    "ddc:820": "ARTS",
    "ddc:830": "ARTS",
    "ddc:840": "ARTS",
    "ddc:850": "ARTS",
    "ddc:860": "ARTS",
    "ddc:870": "ARTS",
    "ddc:880": "ARTS",
    "ddc:890": "ARTS",
    "ddc:900": "SOCI",
    "ddc:910": "SOCI",
    "ddc:920": "ARTS",
    "ddc:930": "ARTS",
    "ddc:940": "ARTS",
    "ddc:944": "ARTS",
    "ddc:950": "ARTS",
    "ddc:960": "ARTS",
    "ddc:970": "ARTS",
    "ddc:980": "ARTS",
    "ddc:990": "ARTS"
}

# map ddc to new scopus
def map_ddc_to_discipline_scopus_new(ddc):
    to_return = ddc_scopus_new.get(ddc, None)
    if to_return is None:
        print("Not found: ", ddc)
    return to_return

### Prepare master file

In [ ]:
theses = pd.read_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-lighter.parquet", engine="pyarrow")
# make oai_set_specs a list
theses["oai_set_specs"] = theses["oai_set_specs"].apply(lambda x: str(x).split("||"))
# explode the dataframe
theses = theses.explode("oai_set_specs")
# apply the mapping function to the discipline column
theses["oai_set_specs"] = theses["oai_set_specs"].apply(map_ddc_to_discipline_scopus)
# Remove rows with no discipline
theses = theses[theses["oai_set_specs"].notna()]
# merge back the dataframe using nnt
theses = theses.groupby(["nnt", "auteur.idref", "auteur.nom", "auteur.prenom", "date_soutenance", "nb_sup"]).agg(
    {"oai_set_specs": lambda x: set(x)}).reset_index()

In [ ]:
# save the dataframe
theses.to_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-lighter-multi.parquet",
                   engine="pyarrow")

In [ ]:
len(theses), len(theses["nnt"])

In [ ]:
sup_color_dict = ["black", "#598dff", "#2a25a2", "#d06c2c", "#61047d", "#cb46a9", "#7450c6", "#ceb300"]
import plotly.express as px
disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENGI', 'CHEM', 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI', 'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']
colors = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Set3
# build dictionary
disciplines_dict = {}
for i, discipline in enumerate(disciplines):
    disciplines_dict[discipline] = colors[i]

print(disciplines_dict)

In [ ]:
# unique values of nb_sup
theses["nb_sup"].unique()

In [ ]:
# shorten the date_soutenance to only the year
theses["date_soutenance"] = theses["date_soutenance"].astype(str).str.split('-').str[0]

### Plot NB_SUP distribution per year

In [ ]:
# for each discipline in disciplines, plot the nb_sup distribution as a stack bar graph per year
import plotly.graph_objects as go
for i, discipline in enumerate(disciplines):
    # Filter the dataframe for the current discipline
    discipline_df = theses[theses["oai_set_specs"].apply(lambda x: discipline in x)].copy(deep=True)
    discipline_df["oai_set_specs"] = discipline
    figure = go.Figure()
    # Group by year and and oai_set_specs and count the number of theses
    discipline_time_evolution = discipline_df
    discipline_time_evolution["counts"] = 1
    discipline_time_evolution = discipline_df.groupby(["date_soutenance", "nb_sup"]).count().reset_index()
    # sort the dataframe by oai_set_specs
    discipline_time_evolution.sort_values(by=["date_soutenance","nb_sup"], inplace=True)
    # add one trace per oai_set_specs
    nb_sup = discipline_time_evolution["nb_sup"].unique()
    # sort the nb_sup
    nb_sup = sorted(list(nb_sup))
    for i, sup in enumerate(nb_sup):
        trace_df = discipline_time_evolution[discipline_time_evolution["nb_sup"] == sup]
        figure.add_trace(go.Bar(
            x=trace_df["date_soutenance"],
            y=trace_df["counts"],
            name=f"{discipline} - {sup}",
            marker_color=sup_color_dict[sup]
        ))
    figure.update_layout(
        height=700,
        title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
        xaxis_title="Year",
        yaxis_title="Number of Theses",
        template="plotly"
    )
    figure.update_xaxes(categoryorder='category ascending')
    figure.update_layout(barmode='stack')
    figure.write_html(f"./results_new/sups_evo/temporal_evolution_multi_{discipline}_sup_bar.html")
    figure.write_image(f"./results_new/sups_evo/temporal_evolution_multi_{discipline}_sup_bar.png")
    # Same plot but with lines
    figure = go.Figure()
    for i, sup in enumerate(nb_sup):
        trace_df = discipline_time_evolution[discipline_time_evolution["nb_sup"] == sup]
        figure.add_trace(go.Scatter(
            x=trace_df["date_soutenance"],
            y=trace_df["counts"],
            connectgaps=True,
            mode='lines+markers',
            name=f"{discipline} - {sup}",
            line=dict(color=sup_color_dict[sup])
        ))
    figure.update_layout(
        height=700,
        title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
        xaxis_title="Year",
        yaxis_title="Number of Theses",
        template="plotly"
    )
    figure.update_xaxes(categoryorder='category ascending')
    figure.write_html(f"./results_new/sups_evo/temporal_evolution_multi_{discipline}_sup_line.html")
    figure.write_image(f"./results_new/sups_evo/temporal_evolution_multi_{discipline}_sup_line.png")

In [ ]:
# make a bar plot for the number of theses per nb_sup
figure = go.Figure()
# Group by year and nb_sup and count the number of theses
discipline_time_evolution = theses
discipline_time_evolution["count"] = 1
discipline_time_evolution = discipline_time_evolution.groupby(["date_soutenance", "nb_sup"]).count().reset_index()
# sort the dataframe by nb_sup
discipline_time_evolution.sort_values(by=["nb_sup"], inplace=True)
# add one trace per oai_set_specs
nb_sup = discipline_time_evolution["nb_sup"].unique()
# sort the nb_sup
nb_sup = sorted(list(nb_sup))
for i, sup in enumerate(nb_sup):
    trace_df = discipline_time_evolution[discipline_time_evolution["nb_sup"] == sup]
    figure.add_trace(go.Bar(
        x=trace_df["date_soutenance"],
        y=trace_df["count"],
        name=f"{sup}",
        marker_color=sup_color_dict[sup]
    ))
figure.update_layout(
    height=700,
    title=f"Temporal Evolution of Multi-Disciplinary Theses",
    xaxis_title="Year",
    yaxis_title="Number of Theses",
    template="plotly"
)
figure.update_xaxes(categoryorder='category ascending')
figure.update_layout(barmode='stack')
figure.write_html(f"./results_new/sups_evo/temporal_evolution_multi_sup_bar.html")
figure.write_image(f"./results_new/sups_evo/temporal_evolution_multi_sup_bar.png")

### Plot temporal evolution of multi-disciplinary theses

In [ ]:
theses = pd.read_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-lighter-multi.parquet", engine="pyarrow")
# shorten the date_soutenance to only the year
theses["date_soutenance"] = theses["date_soutenance"].astype(str).str.split('-').str[0]
# bi-disciplinary theses
theses = theses[theses["oai_set_specs"].apply(lambda x: len(x) == 2)]

In [ ]:
theses.columns

In [ ]:
theses.head(2)

In [ ]:
disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENGI', 'CHEM', 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI',
                'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']
len(disciplines)

In [ ]:
for discipline in disciplines:
    print(discipline)
    # Filter the dataframe for the current discipline
    discipline_df = theses[theses["oai_set_specs"].apply(lambda x: discipline in x)].copy(deep=True)
    discipline_df["oai_set_specs"] = discipline_df["oai_set_specs"].apply(
        lambda x: [i for i in x if i != discipline][0])
    figure_line = go.Figure()
    figure_bar = go.Figure()
    # Group by year and oai_set_specs and count the number of theses
    discipline_time_evolution = discipline_df
    discipline_time_evolution["counts"] = 1
    discipline_time_evolution = discipline_df.groupby(["date_soutenance", "oai_set_specs"]).count().reset_index()
    # sort the dataframe by oai_set_specs
    discipline_time_evolution.sort_values(by=["date_soutenance","oai_set_specs"], inplace=True)
    # add one trace per oai_set_specs
    oai_set_specs = discipline_time_evolution["oai_set_specs"].unique()
    # sort the oai_set_specs
    oai_set_specs = sorted(list(oai_set_specs))
    # sum counts to deduce percentage
    discipline_time_evolution["counts"] = discipline_time_evolution["counts"] / \
        discipline_time_evolution.groupby("date_soutenance")["counts"].transform("sum")
    for i, oai_set_spec in enumerate(oai_set_specs):
        trace_df = discipline_time_evolution[discipline_time_evolution["oai_set_specs"] == oai_set_spec]
        # figure_line.add_trace(go.Scatter(
        #     x=trace_df["date_soutenance"],
        #     y=trace_df["counts"],
        #     connectgaps=False,
        #     mode='lines+markers',
        #     name=f"{discipline} - {oai_set_spec}",
        #     line=dict(color=disciplines_dict[oai_set_spec])
        # ))
        figure_bar.add_trace(go.Bar(
            x=trace_df["date_soutenance"],
            y=trace_df["counts"],
            name=f"{discipline} - {oai_set_spec}",
            marker_color=disciplines_dict[oai_set_spec]
        ))
    # figure_line.update_layout(
    #     height=700,
    #     title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
    #     xaxis_title="Year",
    #     yaxis_title="Number of Theses",
    #     template="plotly"
    # )
    # figure_line.update_xaxes(categoryorder='category ascending')
    # figure_line.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}.png")
    # figure_line.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}.html")
    figure_bar.update_layout(
        height=700,
        title=f"Temporal Evolution of Multi-Disciplinary {discipline} Theses",
        xaxis_title="Year",
        yaxis_title="Number of Theses",
        template="plotly"
    )
    figure_bar.update_xaxes(categoryorder='category ascending')
    figure_bar.update_layout(barmode='stack')
    figure_bar.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}_bar%.png")
    figure_bar.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_{discipline}_bar%.html")

In [ ]:
# joint figure of all disciplines pairs found
figure_line = go.Figure()
figure_bar = go.Figure()
# Group by year and oai_set_specs and count the number of theses
discipline_time_evolution = theses
discipline_time_evolution["counts"] = 1

theses["oai_set_specs"] = theses["oai_set_specs"].apply(lambda x: "-".join(sorted(list(x))))
discipline_time_evolution = theses.groupby(["date_soutenance", "oai_set_specs"]).count().reset_index()
# sort the dataframe by date then oai_set_specs
def custom_sort(x):
    return len(x), x
discipline_time_evolution.sort_values(by=["date_soutenance","oai_set_specs"], inplace=True)
# add one trace per oai_set_specs
oai_set_specs = discipline_time_evolution["oai_set_specs"].unique()
# sort the oai_set_specs
oai_set_specs = sorted(list(oai_set_specs), key=custom_sort)
# sum counts to deduce percentage
discipline_time_evolution["counts"] = discipline_time_evolution["counts"] / \
    discipline_time_evolution.groupby("date_soutenance")["counts"].transform("sum")
for i, oai_set_spec in enumerate(oai_set_specs):
    trace_df = discipline_time_evolution[discipline_time_evolution["oai_set_specs"] == oai_set_spec]
    # figure_line.add_trace(go.Scatter(
    #     x=trace_df["date_soutenance"],
    #     y=trace_df["counts"],
    #     connectgaps=False,
    #     mode='lines+markers',
    #     name=f"{oai_set_spec}",
    #     # line=dict(color=disciplines_dict[oai_set_spec])
    # ))
    figure_bar.add_trace(go.Bar(
        x=trace_df["date_soutenance"],
        y=trace_df["counts"],
        name=f"{oai_set_spec}",
        # marker_color=disciplines_dict[oai_set_spec]
    ))
# figure_line.update_layout(
#     height=700,
#     title=f"Temporal Evolution of Multi-Disciplinary Theses",
#     xaxis_title="Year",
#     yaxis_title="Number of Theses",
#     template="plotly"
# )
# figure_line.update_xaxes(categoryorder='category ascending')
# figure_line.show()
figure_bar.update_layout(
    height=700,
    title=f"Temporal Evolution of Multi-Disciplinary Theses",
    xaxis_title="Year",
    yaxis_title="Number of Theses",
    template="plotly"
)
figure_bar.update_xaxes(categoryorder='category ascending')
figure_bar.update_layout(barmode='stack')
figure_bar.show()

In [ ]:
# save the figures
# figure_line.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_all_line.png")
# figure_line.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_all_line.html")
figure_bar.write_image(f"./results_new/disc_pairs/temporal_evolution_multi_all_bar%.png")
figure_bar.write_html(f"./results_new/disc_pairs/temporal_evolution_multi_all_bar%.html")

## Visualize discipline collaboration

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# read the dataframe
theses = pd.read_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues-lighter-multi.parquet", engine="pyarrow")
# shorten the date_soutenance to only the year
theses["date_soutenance"] = theses["date_soutenance"].astype(str).str.split('-').str[0]
# bi-disciplinary theses
theses = theses[theses["oai_set_specs"].apply(lambda x: len(x) == 2)]
# sort oai_set_specs
theses["oai_set_specs"] = theses["oai_set_specs"].apply(lambda x: sorted(list(x)))
# break oai_set_specs into 2 columns
theses[["disc1", "disc2"]] = pd.DataFrame(theses["oai_set_specs"].tolist(), index=theses.index)

In [ ]:
len(theses)

In [ ]:
disciplines_matched = theses["disc1"].unique()
disciplines_matched = np.concatenate([disciplines_matched, theses["disc2"].unique()])
disciplines_matched = np.unique(disciplines_matched)
disciplines_matched = sorted(list(disciplines_matched))
print(disciplines_matched)

In [ ]:
# Build matrix of disciplines per year
years = sorted(theses["date_soutenance"].unique())
data = [np.zeros((len(disciplines_matched), len(disciplines_matched))) for _ in years]
# Build index dict from disciplines
disciplines_dict = {}
for i, discipline in enumerate(disciplines_matched):
    disciplines_dict[discipline] = i
# Fill matrices
for i, year in enumerate(years):
    # Filter the dataframe for the current year
    discipline_df = theses[theses["date_soutenance"] == year]
    # Group by year and oai_set_specs and count the number of theses
    discipline_time_evolution = discipline_df
    discipline_time_evolution["counts"] = 1
    discipline_time_evolution = discipline_df.groupby(["disc1", "disc2"]).count().reset_index()
    # sort the dataframe by oai_set_specs
    discipline_time_evolution.sort_values(by=["disc1", "disc2"], inplace=True)
    # add one trace per oai_set_specs
    for index, row in discipline_time_evolution.iterrows():
        d1, d2 = row["disc1"], row["disc2"]
        # get the index of the disciplines
        d1_index = disciplines_dict[d1]
        d2_index = disciplines_dict[d2]
        # fill the matrix
        # data[i][d1_index][d2_index] += row["counts"]
        data[i][d2_index][d1_index] += row["counts"]

In [ ]:
# make 0 values nan
for i, matrix in enumerate(data):
    matrix[matrix == 0] = np.nan

In [ ]:
all_data = np.concatenate(data, axis=0)
unique_values = np.unique(all_data)
# Get the 90th percentile of the data
zmax = np.percentile(unique_values, 80)

In [ ]:
import plotly.express as px
colorscale = px.colors.sequential.Viridis

In [ ]:
# Initialize the heatmap figure
fig = go.Figure(
    data=go.Heatmap(
        z=data[0],
        x=disciplines_matched,
        y=disciplines_matched,
        colorscale=colorscale,
        zmin=1,
        zmax=zmax
    ),
    layout=go.Layout(
        title="Discipline Co-occurrence Over Time",
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[dict(label="Play", method="animate", args=[None])]
        )]
    ),
    frames=[
        go.Frame(
            data=[go.Heatmap(
                z=matrix,
                x=disciplines_matched,
                y=disciplines_matched,
                colorscale=colorscale,
                zmin=1,
                zmax=zmax
            )],
            name=str(year)
        )
        for matrix, year in zip(data, years)
    ]
)

# Add slider for year selection
fig.update_layout(
    sliders=[{
        "steps": [
            {
                "args": [[str(year)], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate"}],
                "label": str(year),
                "method": "animate"
            } for year in years
        ],
        "transition": {"duration": 0},
        "x": 0.1,
        "y": -0.1,
        "currentvalue": {"prefix": "Year: "}
    }],
    template="plotly"
)

fig.update_layout(height=700, width=700)
fig.show()

In [ ]:
# save the figure
fig.write_image(f"./results_new/discipline_collab.png")
fig.write_html(f"./results_new/discipline_collab.html")

## View all disciplines in theses

In [ ]:
import pandas as pd
theses = pd.read_parquet(r"C:\Users\sayfe\Desktop\PER\multdisciplinaryOnlineTool\data\theses-soutenues.parquet", engine="pyarrow")

In [ ]:
from unidecode import unidecode

In [ ]:
disciplines = theses["discipline"]
# remove non string values
disciplines = disciplines[disciplines.apply(lambda x: isinstance(x, str))]
# unidecode
disciplines = disciplines.apply(lambda x: unidecode(str(x)))
disciplines = disciplines.apply(lambda x: str(x).lower().split("("))
disciplines = disciplines.apply(lambda x: str(x).replace("(","").replace(")","").split(" et ")).explode()
disciplines = disciplines.apply(lambda x: str(x).replace("[","").replace("]","").split(". ")).explode()
disciplines = disciplines.apply(lambda x: str(x).split(", ")).explode()
disciplines = disciplines.apply(lambda x: str(x).split(" : ")[0]).explode()
disciplines = disciplines.apply(lambda x: str(x).split(":")[0]).explode()
disciplines = disciplines.apply(lambda x: str(x).split(" - ")[0]).explode()
disciplines = disciplines.apply(lambda x: str(x).split("-")[0]).explode()
disciplines = disciplines.apply(lambda x: str(x).split(" & ")[0]).explode()
disciplines = disciplines.apply(lambda x: str(x).split(" / ")).explode()
disciplines = disciplines.apply(lambda x: str(x).strip()).explode()
disciplines = disciplines.apply(lambda x: str(x).replace("'","").replace("'","").replace("’","").replace("’","").replace("’","").replace("’","").replace("’","").replace('"','').replace("  ","").replace(": ",""))
# replace special characters
disciplines = disciplines.apply(lambda x: str(x).replace("é","e").replace("è","e").replace("ê","e").replace("ë","e").replace("î","i").replace("ï","i").replace("ô","o").replace("ö","o").replace("û","u").replace("ü","u")).unique()
len(disciplines), sorted(disciplines)

In [ ]:
# write list to file
with open("./results_new/disciplines.txt", "w", encoding="utf-8") as f:
    for discipline in sorted(disciplines):
        f.write(f"{discipline}\n")